In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
    fix_units,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is sharing one row (see [](general:ic)) if the measurement was taken on the same day. No further action is nessary.

In [ ]:
split_data(data, ["donor_et_dso", "donor_et_id_et"])
assert (
    data.loc[:, ["sampling_date_et", "sampling_date_dso"]]
    .diff(axis=1)
    .iloc[:, 1]
    .dropna()
    == 0
).all(), "Dates sometimes different"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). The following analysis compares the data from the different sources. All rows were kept.

In [ ]:
data["Institute with a measurement date"] = (
    (~data["sampling_date_dso"].isna()) + (~data["sampling_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = donors[donors.isin(targetpop["donor_et_id_et"])]
data["sampling_date"] = collapse_col(
    data.loc[:, ["sampling_date_dso", "sampling_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "sampling_date",
    "Institute with a measurement date",
)
data.drop(
    columns=["sampling_date", "donor", "Institute with a measurement date"],
    inplace=True,
)

### Unit Conversions

First common translations were applied and then we converted different pressure measurements to a common unit (see [](general:uc)). Afterwards, unit specifier columns with only a single unit were removed.

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
fix_units(
    data,
    "p_co2_mm_hg_et",
    "p_co2_unit_et",
    config["data"]["unit_conversions"]["gas_mmhg"]["target"],
    config["data"]["unit_conversions"]["gas_mmhg"]["factors"],
)
fix_units(
    data,
    "p_co2_o2_100_mm_hg_et",
    "p_co2_o2_100_unit_et",
    config["data"]["unit_conversions"]["gas_mmhg"]["target"],
    config["data"]["unit_conversions"]["gas_mmhg"]["factors"],
)
fix_units(
    data,
    "p_o2_mm_hg_et",
    "p_o2_unit_et",
    config["data"]["unit_conversions"]["gas_mmhg"]["target"],
    config["data"]["unit_conversions"]["gas_mmhg"]["factors"],
)
fix_units(
    data,
    "p_o2_o2_100_mm_hg_et",
    "p_o2_o2_100_unit_et",
    config["data"]["unit_conversions"]["gas_mmhg"]["target"],
    config["data"]["unit_conversions"]["gas_mmhg"]["factors"],
)

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
# Another base class might be necessary, see util.py
# describe columns, without checks for now, order is important
class DonorPostmortemLabBloodGases(SpenderID):
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    excess_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Excess",
        description="Excess measured in mmol/l",
    )
    excess_o2_o2_100_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Excess at 100% pO2",
        description="Excess measured in mmol/l at 100% pO2",
    )
    hco3_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HCO3",
        description="HCO3 measured in mmol/l",
    )
    hco3_o2_100_mmol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HCO3 at 100% pO2",
        description="HCO3 measured in mmol/l at 100% pO2",
    )
    o2_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="O2",
        description="O2 concentration in %",
    )
    o2_saturation_o2_100_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="O2 Saturation at 100% pO2",
        description="O2 saturation in % at 100% pO2",
    )
    o2_saturation_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="O2 Saturation",
        description="O2 saturation in %",
    )
    p_co2_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pCO2",
        description="pCO2 in mmHg",
    )
    p_co2_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pCO2 at 100% pO2",
        description="pCO2 in mmHg at 100% pO2",
    )
    blood_ph: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pH",
        description="pH in mmHg",
    )
    blood_ph: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pH at 100% pO2",
        description="pH in mmHg at 100% pO2",
    )
    p_o2_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pO2",
        description="pO2 in mmHg",
    )
    p_o2_mm_hg: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="pO2 at 100% pO2",
        description="pO2 in mmHg at 100% pO2",
    )
    peep_cm_h2o: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="PEEP",
        description="pO2 in cmH20",
    )
    peep_cm_h2o: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="PEEP",
        description="pO2 in cmH20",
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )
    sample_tissue: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tissue",
        description="From what tissue was the sample taken?",
        isin=["Heparinblut", "Blut", "Biopsie", "Serum", "Katheterurin", "Liquor"],
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="Date when the sample was taken",
    )

    class Config:
        title = "Donor Postmortem Blood Gas Lab Dataset"
        description = "Each row represents a blood gas lab test for a deceased donor. The data is based on the 'element_spender_postmortem_labor_blutgase.csv' file. It contains data from the ET and DSO."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabBloodGases, data)

In [ ]:
DonorPostmortemLabBloodGases.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabBloodGases.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)